# Прогнозирование медианной стоимости жилья

**Цель проекта:**
Разработать и оценить модели линейной регрессии для прогнозирования медианной стоимости жилья (`median_house_value`) в жилых массивах Калифорнии на основе демографических и географических характеристик в Apache Spark(PySpark) с использованием Spark ML.

**Данные:**
В работе используется набор данных по жилым массивам штата Калифорния за 1990 год, содержащий числовые признаки (например, координаты, медианный возраст жилья, число комнат и домохозяйств, численность населения, медианный доход) и категориальный признак ocean_proximity (близость к океану). 

**Этапы проекта:**
- **Загрузка и предобработка данных:** первичная проверка данных на соответствие типов, обработка пропусков. Кодирование и масштабирование признаков.
- **Обучение моделей:** обучение моделей линейной регрессии на всех имеющихся признаках и только на количественных.
- **Оценка качества и анализ результатов:** сравнение моделей и формулировка выводов на основании метрик RMSE, MAE и R2.

In [ ]:
import pandas as pd 

import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.types import *
import pyspark.sql.functions as F

from pyspark.ml.feature import (StringIndexer,
                                VectorAssembler,
                                StandardScaler,
                                OneHotEncoder)
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

RANDOM_SEED = 42


Для выполнения данного проекта запустим Spark в локальном (local mode) режиме, который обработает код без распределения вычислений, используя мощности одного компьютера. 



In [8]:
spark = SparkSession\
                    .builder \
                    .master('local') \
                    .appName('Cost_Prediction_California_Housing') \
                    .getOrCreate()


25/12/17 15:42:38 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## Загрузка и предобработка данных

Прочтем данные с помощью метода spark.read.load(), который сразу вернет Spark DataFrame. Атрибуты `inferSchema=True` и `header=True` позволяют автоматически распознать типы данных и сделать первую строку названиями колонок. 

С помощью метода действия show(), отобразим первые 5 строк данных. 

In [ ]:
df_housing = spark.read.load('data/housing.csv', 
                             format="csv", 
                             sep=",", 
                             inferSchema=True,
                             header=True
                            )
df_housing.show(5)


+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|longitude|latitude|housing_median_age|total_rooms|total_bedrooms|population|households|median_income|median_house_value|ocean_proximity|
+---------+--------+------------------+-----------+--------------+----------+----------+-------------+------------------+---------------+
|  -122.23|   37.88|              41.0|      880.0|         129.0|     322.0|     126.0|       8.3252|          452600.0|       NEAR BAY|
|  -122.22|   37.86|              21.0|     7099.0|        1106.0|    2401.0|    1138.0|       8.3014|          358500.0|       NEAR BAY|
|  -122.24|   37.85|              52.0|     1467.0|         190.0|     496.0|     177.0|       7.2574|          352100.0|       NEAR BAY|
|  -122.25|   37.85|              52.0|     1274.0|         235.0|     558.0|     219.0|       5.6431|          341300.0|       NEAR BAY|
|  -122.25|   37.85|              

In [11]:
print(pd.DataFrame(df_housing.dtypes, columns=['column', 'type']))


               column    type
0           longitude  double
1            latitude  double
2  housing_median_age  double
3         total_rooms  double
4      total_bedrooms  double
5          population  double
6          households  double
7       median_income  double
8  median_house_value  double
9     ocean_proximity  string


Spark корректно распознал типы данных: данные содержат одну категориальную колонку - `ocean_proximity`, остальные - количественные. При этом для всех количественных данных утсановлен тип - double, т.е. число с плавающей точкой (аналог float-типа в python), даже если логически это должно быть целое число  (например, общее количество комнат, спален и тп). В данном случае это не принципиально, поэтому оставим типы как есть. 

Учитывая значения в поле `median_income` можно предположить, что доход указан в десятках тысяч долларов, т.е. 8,3 - это 83 тыс.долл.

In [12]:
df_housing.describe().show(truncate=False)


25/12/17 17:38:41 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-------------------+-----------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+---------------+
|summary|longitude          |latitude         |housing_median_age|total_rooms       |total_bedrooms    |population        |households       |median_income     |median_house_value|ocean_proximity|
+-------+-------------------+-----------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+---------------+
|count  |20640              |20640            |20640             |20640             |20433             |20640             |20640            |20640             |20640             |20640          |
|mean   |-119.56970445736148|35.6318614341087 |28.639486434108527|2635.7630813953488|537.8705525375618 |1425.4767441860465|499.5396802325581|3.8706710029070246|206855.81690891474|NULL           |
|stddev |2.003531723

На основании статистической сводки по данным можно сделать следующие выводы: 
- Данные содержат 20640 записей, из них пропуски обнаружены только в поле `total_bedrooms` (~ 1%). При этом среднее значение в этом поле (mean ~537.87) очень близко к стандартному отклонению(stddev ~421.39), что свидетельствует о высокой вариантивности в данных - возможны выбросы. В этом случае для заполнения пропусков в даннных лучше подходит медиана, т.к. она более устойчива к выбросам.
- Количественные признаки лежат в разных диапазонах, поэтому для исключения влияние неоднородности данных необходимо провести масштабирование. 

Разделим данные на обучающую и тестовую выборки с помощью метода randomSplit(). Затем рассчитаем медиану по обучающей выборке и применим ее к нашим данным.

In [ ]:
train_data, test_data = df_housing.randomSplit([.8,.2], seed=RANDOM_SEED)
print(train_data.count(), test_data.count()) 


16560 4080


In [15]:
#Вычисляем медиану на обучающей выборке
median_value = train_data.approxQuantile('total_bedrooms', [0.5], 0.01)[0]
median_value


430.0

In [16]:
train_data = train_data.fillna({'total_bedrooms': median_value})
test_data = test_data.fillna({'total_bedrooms': median_value})


Для удобства подготовки данных разделим колонки на два типа: числовые и текстовые, которые представляют категориальные данные, отдельно выделим колонку с целевой переменной. 

In [18]:
categorical_cols = ['ocean_proximity']
numerical_cols  = ['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', \
                   'population', 'households', 'median_income']
target = 'median_house_value' 


Для начала трансформируем категориальные признаки в обучающей и тестовой выборке с помощью трансформера StringIndexer, который приводит текстовые значения к числовым. Чтобы трансформировать таблицу, нужно сначала передать её в метод fit() и запустить трансформацию методом transform(). В терминах ленивых вычислений, которые применяются в MLlib, fit() — это транcформация, а transform() — это действие.

In [19]:
indexer = StringIndexer(inputCols=categorical_cols, 
                        outputCols=[c+'_idx' for c in categorical_cols]) 
indexer_model = indexer.fit(train_data)
train_data = indexer_model.transform(train_data)
test_data = indexer_model.transform(test_data)

cols = [c for c in train_data.columns for i in categorical_cols if (c.startswith(i))]
train_data.select(cols).show(3) 


+---------------+-------------------+
|ocean_proximity|ocean_proximity_idx|
+---------------+-------------------+
|     NEAR OCEAN|                2.0|
|     NEAR OCEAN|                2.0|
|     NEAR OCEAN|                2.0|
+---------------+-------------------+
only showing top 3 rows


Дополнительно применим стандартное OHE-кодирование.

In [20]:
encoder = OneHotEncoder(inputCols=[c+'_idx' for c in categorical_cols],
                        outputCols=[c+'_ohe' for c in categorical_cols])
encoder_model = encoder.fit(train_data)
train_data = encoder_model.transform(train_data)
test_data = encoder_model.transform(test_data)

cols = [c for c in train_data.columns for i in categorical_cols if (c.startswith(i))]
train_data.select(cols).show(3) 


+---------------+-------------------+-------------------+
|ocean_proximity|ocean_proximity_idx|ocean_proximity_ohe|
+---------------+-------------------+-------------------+
|     NEAR OCEAN|                2.0|      (4,[2],[1.0])|
|     NEAR OCEAN|                2.0|      (4,[2],[1.0])|
|     NEAR OCEAN|                2.0|      (4,[2],[1.0])|
+---------------+-------------------+-------------------+
only showing top 3 rows


Т.к. у нас в данных только один категориальный признак, то объединение в единый вектор с помощью VectorAssembler не требуется. Используем колонку `ocean_proximity_ohe` при подготовке финального списка признаков.

Далее применим масштабирование к количественным признакам. Для этого предварительно соберем выбранные числовые признаки в единый вектор значений с помощью метода VectorAssembler(), а затем применим масштабирование с помощью метода StandardScaler().

In [21]:
numerical_assembler = VectorAssembler(inputCols=numerical_cols,outputCol="numerical_features")
train_data = numerical_assembler.transform(train_data)
test_data = numerical_assembler.transform(test_data)


In [22]:
scaler = StandardScaler(inputCol='numerical_features',
                        outputCol="numerical_features_scaled")
scaler_model = scaler.fit(train_data)
train_data = scaler_model.transform(train_data)
test_data = scaler_model.transform(test_data)


In [23]:
print(train_data.columns)
print(test_data.columns)


['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value', 'ocean_proximity', 'ocean_proximity_idx', 'ocean_proximity_ohe', 'numerical_features', 'numerical_features_scaled']
['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value', 'ocean_proximity', 'ocean_proximity_idx', 'ocean_proximity_ohe', 'numerical_features', 'numerical_features_scaled']


**Вывод по разделу "Загрузка и предобработка данных":**

1. Выборка разделена на обучающую и тестовую.
2. Пропуски в колонке `total_bedrooms` заменены на медианное значение, полученное на обучающей выборке.
2. Категориальный признак 'ocean_proximity' приведен к числовым значениям с помощью метода StringIndexer() и закодирован с помощью метода OneHotEncoder().
3. Количественные признаки масштабированы с помощью метода StandartScaler.

## Обучение моделей

### Модель №1


Обучим модель LinearRegression на всех имеющихся признаках, которые мы предварительно подготовили (кодирование, масштабирование).

Соберём трансформированные категорийные и числовые признаки в единый вектор, который будет подаваться на вход модели, с помощью VectorAssembler().

In [24]:
all_features = ['ocean_proximity_ohe','numerical_features_scaled']

final_assembler = VectorAssembler(inputCols=all_features, outputCol="features") 
train_data = final_assembler.transform(train_data)
test_data = final_assembler.transform(test_data)

display(train_data.select(all_features).show(3)) 
test_data.select(all_features).show(3)


+-------------------+-------------------------+
|ocean_proximity_ohe|numerical_features_scaled|
+-------------------+-------------------------+
|      (4,[2],[1.0])|     [-61.931952286653...|
|      (4,[2],[1.0])|     [-61.907050013920...|
|      (4,[2],[1.0])|     [-61.892108650280...|
+-------------------+-------------------------+
only showing top 3 rows


None

+-------------------+-------------------------+
|ocean_proximity_ohe|numerical_features_scaled|
+-------------------+-------------------------+
|      (4,[2],[1.0])|     [-61.907050013920...|
|      (4,[2],[1.0])|     [-61.872186832094...|
|      (4,[2],[1.0])|     [-61.872186832094...|
+-------------------+-------------------------+
only showing top 3 rows


Инициализируем модель. Укажем колонку с целевой переменной и все имеющиеся признаки для обучения.

In [25]:
lr_1 = LinearRegression(labelCol=target, featuresCol='features')

model_1 = lr_1.fit(train_data)


25/12/17 18:07:52 WARN Instrumentation: [79c66201] regParam is zero, which might cause numerical instability and overfitting.
25/12/17 18:07:52 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
25/12/17 18:07:53 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


Модель — это оценщик в терминологии сущностей библиотеки MLlib. Оценщик возвращает трансформер с методом transform(), который в свою очередь возвращает трансформированную таблицу с колонкой предсказания модели. Таким образом, для получения предсказаний мы применим метод trandform()полученной модели к тестовым данным.

In [26]:
predictions_1 = model_1.transform(test_data)

predictedLabes_1 = predictions_1.select('median_house_value', 'prediction')
predictedLabes_1.show(10) 


+------------------+------------------+
|median_house_value|        prediction|
+------------------+------------------+
|          103600.0|150538.00309671462|
|          106700.0|217632.03521285625|
|           73200.0|125172.05474225292|
|           90100.0| 195024.6803732384|
|           67000.0|152295.04722368577|
|           86400.0|186081.34313048003|
|           70500.0| 163946.1746366974|
|           85100.0|180016.90879898006|
|           80500.0|181585.35948785022|
|           96000.0|  170320.953513565|
+------------------+------------------+
only showing top 10 rows


По условиям проекта модель необходиом оценить с помощью метрик RMSE, MAE и R2. Для этих целей применим стандартный трансформер RegressionEvaluator(). 

In [30]:
rmse = RegressionEvaluator(labelCol='median_house_value',
                           predictionCol='prediction',
                           metricName='rmse').evaluate(predictions_1)
mae  = RegressionEvaluator(labelCol='median_house_value',
                           predictionCol='prediction',
                           metricName='mae').evaluate(predictions_1)
r2   = RegressionEvaluator(labelCol='median_house_value',
                           predictionCol='prediction',
                           metricName='r2').evaluate(predictions_1)

print('RMSE:', rmse)
print('MAE:', mae)
print('R2:', r2)


RMSE: 70787.0224384578
MAE: 50864.15060034779
R2: 0.6378395213725729


Более робастная MAE ниже RMSE, следовательно, можно предположить, что в данных присутствуют выбросы. В среднем модель отклоняется от истинного значения целевого признака на ≈50000 у.е. 

Коэффициент детерминации R2 показывает, что в ~ 64% случаев предсказание модели ближе к истине, чем среднее значение целевого признака.

### Модель №2 

Обучим модель LinearRegression только на количественных признаках, которые мы предварительно подготовили (масштабирование).

Объединять данные, как в предыдущем случае не потребуется. При инициализации модели укажем вектор с количественными признаками на входе модели: `numerical_features_scaled`. Таке данные уже поделены на обучающую и тестовую выборку.

In [31]:
lr_2 = LinearRegression(labelCol=target, featuresCol='numerical_features_scaled')

model_2 = lr_2.fit(train_data) 


25/12/17 18:23:33 WARN Instrumentation: [1bdbf19f] regParam is zero, which might cause numerical instability and overfitting.


In [32]:
predictions_2 = model_2.transform(test_data)

predictedLabes_2 = predictions_2.select("median_house_value", "prediction")
predictedLabes_2.show(10) 


+------------------+------------------+
|median_house_value|        prediction|
+------------------+------------------+
|          103600.0|100752.10695006652|
|          106700.0|190794.11885934556|
|           73200.0| 74733.42223240901|
|           90100.0| 162345.7214194457|
|           67000.0|119471.26093414566|
|           86400.0|155925.45490939496|
|           70500.0|131202.85506911343|
|           85100.0|150453.71075923042|
|           80500.0|150170.70999537082|
|           96000.0|133763.13862226438|
+------------------+------------------+
only showing top 10 rows


Рассчитаем метрики для модели №2.

In [ ]:

rmse = RegressionEvaluator(labelCol='median_house_value',
                           predictionCol='prediction',
                           metricName='rmse').evaluate(predictions_2)
mae  = RegressionEvaluator(labelCol='median_house_value',
                           predictionCol='prediction',
                           metricName='mae').evaluate(predictions_2)
r2   = RegressionEvaluator(labelCol='median_house_value',
                           predictionCol='prediction',
                           metricName='r2').evaluate(predictions_2)

print('RMSE:', rmse)
print('MAE:', mae)
print('R2:', r2)


RMSE: 71792.07548703383
MAE: 51805.28978969226
R2: 0.6274824107722436


Коэффициент детерминации R2 уменьшился на 1%, при этом MAE и RMSE увеличились ~ 1000 у.е.

**Вывод по разделу "Обучение моделей":**

По итогам обучения моделей для предсказания медианной стоимости жилья в Калифорнии получили следующие результаты:


| Модель                                       | RMSE     | MAE      | R²   |
| -------------------------------------------- | ---------| -------- | -----|
| **С числовыми и категориальными признаками** | 70787.02 | 50864.15 | 0.64 |
| **Только с числовыми признаками**            | 71792.07 | 51805.30 | 0.63 |
| **Разница**                                  | 1005.05  | 941.15   | 0.01 |


Категориальный признак `ocean_proximity` улучшает качество модели:

- Ошибка RMSE и MAE уменьшается на ~ 1000 у.е.
- Коэффициент детерминации R² повышается на 1%.

Хотя прирост качества невелик, он всё же присутствует. Это говорит о том, что категориальная информация может давать полезный сигнал для предсказания стоимости жилья.

## Общий вывод

В ходе проекта разработаны  модели линейной регрессии для предсказания медианной стоимости жилья в Калифорнии на основе данных 1990 года. 

Модель, использующая как числовые, так и категориальные признаки, показала немного лучшие результаты (R² = 0.64 против 0.63). Это подтверждает значимость включения категориального признака ocean_proximity в процесс обучения. 

В ходе проекта были выполнены следующие этапы:
1. Предобработка данных:
- Выборка разделена на обучающую и тестовую.
- Пропуски в колонке `total_bedrooms` заменены на медианное значение, полученное на обучающей выборке.
- Категориальный признак 'ocean_proximity' приведен к числовым значениям с помощью метода StringIndexer() и закодирован с помощью метода OneHotEncoder().
- Количественные признаки масштабированы с помощью метода StandartScaler.

2. Обучены 2 модели линейной регрессии на всех имеющихся признаках и только на количественных. В результате получены следующие результаты:

| Модель                                       | RMSE     | MAE      | R²   |
| -------------------------------------------- | ---------| -------- | -----|
| **С числовыми и категориальными признаками** | 70787.02 | 50864.15 | 0.64 |
| **Только с числовыми признаками**            | 71792.07 | 51805.30 | 0.63 |
| **Разница**                                  | 1005.05  | 941.15   | 0.01 |

3. Проведен сравнительный анализ полученных метрик, в результате чего пришли к выводу, что использование категориального признака при обучении дает прирост в качестве для всех метрик.

**Возможности для дальнейшего улучшения модели:**
- Применение более сложных моделей (например, решающих деревьев или градиентного бустинга).
- Обработка выбросов в данных.
- Подбор гиперпараметров для моделей.